In [ ]:
import jax.numpy as jnp
import matplotlib.colors as mcolors
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt

from vizopt.templates.color import lab_to_rgb, optimize_colors, plot_colored_words

In [ ]:
words = ["paper", "Papier", "papel", "palpiri", "carta", "karatasi"]
words = [str(i_int) for i_int in range(len(words))]

In [ ]:
def edit_distance(a, b):
    """Levenshtein distance via DP."""
    a, b = a.lower(), b.lower()
    m, n = len(a), len(b)
    dp = list(range(n + 1))
    for i, ca in enumerate(a, 1):
        prev = dp[:]
        dp[0] = i
        for j, cb in enumerate(b, 1):
            dp[j] = prev[j-1] if ca == cb else 1 + min(prev[j], dp[j-1], prev[j-1])
    return dp[n]


In [ ]:
distances = pd.DataFrame(
    [[edit_distance(w1, w2) for w2 in words] for w1 in words],
    index=words,
    columns=words,
)
distances

In [ ]:
main_color = "#784086"

main_color = "#1CB61C"

In [ ]:
sparse_cb = lambda i, loss, *_: print(f"iter {i:4d}  loss={loss:.6f}") if i % 200 == 0 else None

colors, history = optimize_colors(
    distances,
    fixed_colors={words[0]: mcolors.to_rgb(main_color)},
    target_max_delta_e=50.0,
    learning_rate=0.05,
    n_iters=1000,
    callback=sparse_cb,
)
print(f"Final loss: {history[-1]['total']:.6f}")

In [ ]:
from scipy.stats import spearmanr
from vizopt.templates.color import rgb_to_lab

n = len(words)
pairs = [(i, j) for i in range(n) for j in range(i + 1, n)]
D = np.array(distances)

lab_opt = np.array(rgb_to_lab(jnp.array(colors)))
color_dists_opt = [np.linalg.norm(lab_opt[i] - lab_opt[j]) for i, j in pairs]
rho, _ = spearmanr(color_dists_opt, [D[i, j] for i, j in pairs])
print(f"Spearman ρ = {rho:.3f}")

#plot_colored_words(words, colors)

In [ ]:
n_restarts = 3
results = []
for seed in range(n_restarts):
    c, history = optimize_colors(
        distances,
        fixed_colors={words[0]: mcolors.to_rgb(main_color)},
        target_max_delta_e=100.0,
        learning_rate=0.05,
        n_iters=1000,
        seed=seed,
    )
    loss = history[-1]["total"]
    results.append((loss, c))
    print(f"seed={seed:2d}  loss={loss:.6f}")

best_loss, best_colors = min(results, key=lambda x: x[0])
print(f"\nBest loss = {best_loss:.6f}")


In [ ]:
import networkx as nx

In [ ]:
tree = nx.balanced_tree(r=2, h=3, create_using=nx.DiGraph)
tree.nodes

In [ ]:
mapping = {}
queue = [(0, "")]
while queue:
    node, label = queue.pop(0)
    mapping[node] = label
    for i, child in enumerate(tree.successors(node)):
        queue.append((child, label + str(i)))

mapping[0] = "root"
tree = nx.relabel_nodes(tree, mapping)

In [ ]:
nx.draw_networkx(tree)

In [ ]:
from IPython.display import HTML

from vizopt.animation import SnapshotCallback, animate
from vizopt.templates.color import build_color_input_parameters, color_palette_template

In [ ]:
input_parameters = build_color_input_parameters(
    distances,
    fixed_colors={words[0]: mcolors.to_rgb(main_color)},
    target_max_delta_e=50.0,
)
problem = color_palette_template.instantiate(input_parameters)
snapshot_cb = SnapshotCallback(every=50)
optim_vars_opt, history = problem.optimize(n_iters=5000, track_every=100, callback=snapshot_cb)

In [ ]:
from IPython.display import SVG

from vizopt.animation import snapshots_to_animated_svg

svg = snapshots_to_animated_svg(problem, snapshot_cb.snapshots, fps=20)
SVG(data=svg)

In [ ]:
pd.DataFrame(history).set_index("iteration").plot(marker=".")
plt.ylabel("Loss value")
plt.xlabel("Iteration")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from vizopt.templates.color import rgb_to_lab

colormaps = ['Pastel1', 'Pastel2', 'Paired', 'Accent', 'Dark2']

results = {}
for name in colormaps:
    cmap = plt.get_cmap(name)
    n = cmap.N  # number of discrete colors
    rgb = np.array([cmap(i)[:3] for i in range(n)])  # (n, 3) sRGB
    lab = np.array(rgb_to_lab(rgb))                   # (n, 3) LAB
    coverage = -(lab[:, 1].var() + lab[:, 2].var())   # matches _coverage()
    results[name] = coverage
    print(f"{name:10s}  n={n:2d}  coverage={coverage:8.2f}  (a*+b* spread={-coverage:.2f})")


In [ ]:
fig, axes = plt.subplots(1, len(colormaps), figsize=(14, 4), sharex=True, sharey=True)

for ax, name in zip(axes, colormaps):
    cmap = plt.get_cmap(name)
    rgb = np.array([cmap(i)[:3] for i in range(cmap.N)])
    lab = np.array(rgb_to_lab(rgb))
    for r, g, b in zip(rgb, lab[:, 1], lab[:, 2]):
        ax.scatter(b, g, color=r, s=200, edgecolors="0.3", linewidths=0.5)
    ax.set_title(name)
    ax.set_xlabel("b*")
    ax.axhline(0, color="0.8", lw=0.5)
    ax.axvline(0, color="0.8", lw=0.5)

axes[0].set_ylabel("a*")
plt.tight_layout()
plt.show()
